# Cross-disease: OGN+RSPO3+ Fib Immune Neighbors — UC vs Colonic CD vs Ileal CD (Ext Fig 9D)

KNN classifier comparing OGN+RSPO3+ Fib immune neighborhood composition
across UC, colonic CD, and ileal CD at 50/100/200um radii.

In [ ]:
# ── Paths ──
UC_DIR     = "/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged"
CD_DIR     = "/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged"
OUTPUT_DIR = UC_DIR + "/cross_disease_fib_neighbors"

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

from scipy.spatial import cKDTree
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import osfrom scipy.sparse import csr_matrix
from sklearn.neighbors import KNeighborsClassifier
from typing import Iterable, Optional, Dict, Set
import re


In [ ]:
samples = pd.read_csv('/data/ydn4687/ibd/yiming_nat_com_samples.txt',sep='\t')

In [ ]:
wtx_uc = sc.read_h5ad('/path/to/cosmx_data/uc_1k_6k_wtx/whole_trans/Processed_merged/v7/h5ad_obj/qc_wtx_v7_merged_anno.h5ad')


In [ ]:
wtx_cd = sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')

In [ ]:
wtx_uc_inf = wtx_uc[wtx_uc.obs['patient'].isin(['UC_P1', 'UC_P2'])]

In [ ]:
wtx_cd_inf = wtx_cd[wtx_cd.obs['condition']=='Inf']

In [ ]:
wtx_cd_inf.obs[['slide','fov']] .drop_duplicates().groupby("slide").size()

In [ ]:
wtx_cd_inf_imm = wtx_cd_inf[
    wtx_cd_inf.obs['cell_category'].isin(['T', 'Myeloid', 'B', 'Granulocyte']) |
    (
        (wtx_cd_inf.obs['cell_category'] == 'Connective') &
        (wtx_cd_inf.obs['cell_type'].isin(['OGN+RSPO3+ Fib']))
    )
].copy()

In [ ]:
wtx_uc_inf_imm = wtx_uc_inf[
    wtx_uc_inf.obs['aucell_cell_category_from_type'].isin(['T', 'Myeloid', 'B', 'Granulocyte']) |
    (
        (wtx_uc_inf.obs['aucell_cell_category_from_type'] == 'Connective') &
        (wtx_uc_inf.obs['aucell_cell_type_short'].isin(['OGN+RSPO3+ Fib']))
    )
].copy()

In [ ]:

def standardize_celltype(
    label: str,
    collapse_prolif: bool = True,
    collapse_trm_gzmk_order: bool = True,
    extra_map: Optional[Dict[str, str]] = None,
) -> str:
    """
    Standardize immune cell type strings across UC/CD naming conventions.

    Parameters
    ----------
    label : str
        Original cell type label.
    collapse_prolif : bool
        If True, collapses '(Prolif)' variants into a common label (e.g. 'CD8 Trm Prolif').
    collapse_trm_gzmk_order : bool
        If True, makes CD8/CD4 Trm GZMK +/- naming consistent.
    extra_map : dict, optional
        User-provided exact mapping overrides, applied after normalization.

    Returns
    -------
    str
        Canonical/standardized label.
    """
    if label is None:
        return label
    s = str(label).strip()

    # Normalize spacing
    s = re.sub(r"\s+", " ", s)

    # Unify common typos / synonyms
    # (add more if you discover them)
    s = s.replace("Exhaus", "Exhausted")
    s = s.replace("Inf ", "Inflammatory ")
    s = s.replace("Mo-Mac", "Mono/Mac")
    s = s.replace("Trans cDC2/Mac", "Trans cDC2/Mono/Mac")

    # Make CD8/CD4 Trm naming order consistent
    if collapse_trm_gzmk_order:
        # UC style: "GZMK+ CD8 Trm" -> "CD8 GZMK+ Trm"
        m = re.fullmatch(r"(GZMK[+-]) (CD8) Trm", s)
        if m:
            s = f"{m.group(2)} {m.group(1)} Trm"

        m = re.fullmatch(r"(GZMK[+-]) (CD8) Trm\(Prolif\)", s)
        if m:
            s = f"{m.group(2)} {m.group(1)} Trm(Prolif)"

        # UC style: "GZMK- CD8 Trm" might also appear as "GZMK- CD8 Trm " etc.
        # handled above by regex + whitespace normalize

    # Collapse proliferating Trm variants if requested
    if collapse_prolif:
        # CD style: "CD8 GZMK- Trm(Prolif)" -> "CD8 Trm Prolif"
        if re.search(r"\(Prolif\)", s):
            # Keep CD4 vs CD8 if present; otherwise map to generic Cycling T
            if s.startswith("CD8"):
                s = "CD8 Trm Prolif"
            elif s.startswith("CD4"):
                s = "CD4 Trm Prolif"
            else:
                s = "Cycling T"

    # Harmonize a few known label patterns
    # CD: "CD4 Trm+D42" looks like a Trm subtype; align to CD4 Trm unless you truly need D42
    s = re.sub(r"^CD4 Trm\+.*$", "CD4 Trm", s)

    # Harmonize resting T / effector naming a bit
    # (optional: you may prefer to keep these exactly as-is)
    s = s.replace("Resting T", "Resting T")
    s = s.replace("CD4 Effector", "CD4 Effector")
    s = s.replace("CD8 Effector", "CD8 Effector")

    # Apply user overrides last
    if extra_map and s in extra_map:
        s = extra_map[s]

    return s


def align_labels(
    labels: Iterable[str],
    collapse_prolif: bool = True,
    extra_map: Optional[Dict[str, str]] = None,
) -> list:
    """Vectorized wrapper for standardize_celltype."""
    return [standardize_celltype(x, collapse_prolif=collapse_prolif, extra_map=extra_map) for x in labels]


def compare_label_sets(
    cd_labels: Set[str],
    uc_labels: Set[str],
    collapse_prolif: bool = True,
    extra_map: Optional[Dict[str, str]] = None,
):
    """Quick helper to see overlap vs. differences after standardization."""
    cd_std = {standardize_celltype(x, collapse_prolif=collapse_prolif, extra_map=extra_map) for x in cd_labels}
    uc_std = {standardize_celltype(x, collapse_prolif=collapse_prolif, extra_map=extra_map) for x in uc_labels}

    shared = sorted(cd_std & uc_std)
    cd_only = sorted(cd_std - uc_std)
    uc_only = sorted(uc_std - cd_std)

    return {
        "shared": shared,
        "cd_only": cd_only,
        "uc_only": uc_only,
        "n_shared": len(shared),
        "n_cd_only": len(cd_only),
        "n_uc_only": len(uc_only),
    }


In [ ]:
# cd
wtx_cd_inf.obs["celltype_aligned"] = align_labels(
    wtx_cd_inf.obs["cell_type_short"].tolist(),
    collapse_prolif=True
)

# UC
wtx_uc_inf.obs["celltype_aligned"] = align_labels(
    wtx_uc_inf.obs["aucell_cell_type_short"].tolist(),
    collapse_prolif=True
)

In [ ]:
wtx_uc_inf_imm = wtx_uc_inf[
    wtx_uc_inf.obs['aucell_cell_category_from_type'].isin(['T', 'Myeloid', 'B', 'Granulocyte']) |
    (
        (wtx_uc_inf.obs['aucell_cell_category_from_type'] == 'Connective') &
        (wtx_uc_inf.obs['aucell_cell_type_short'].isin(['OGN+RSPO3+ Fib']))
    )
].copy()

In [ ]:
wtx_cd_inf_imm = wtx_cd_inf[
    wtx_cd_inf.obs['cell_category'].isin(['T', 'Myeloid', 'B', 'Granulocyte']) |
    (
        (wtx_cd_inf.obs['cell_category'] == 'Connective') &
        (wtx_cd_inf.obs['cell_type'].isin(['OGN+RSPO3+ Fib']))
    )
].copy()

In [ ]:
summary = compare_label_sets(
    set(wtx_cd_inf_imm.obs["celltype_aligned"].unique()),
    set(wtx_uc_inf_imm.obs["celltype_aligned"].unique()),
    collapse_prolif=True
)
summary["n_shared"], summary["cd_only"], summary["uc_only"]


In [ ]:
shared_imm = summary['shared']

In [ ]:

R_LIST = [25, 50, 100, 200]
CELLTYPE_COL = "celltype_aligned"
FOV_COL = "fov"
SLIDE_COL = "slide"
PATIENT_COL = "patient"

RSPO3_FIB_LABEL = "OGN+RSPO3+ Fib"
CONNECTIVE_LABEL = "Connective"



In [ ]:
def aggregate_fov_to_slide_patient(fov_comp_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Aggregate per-FOV composition to slide and patient using medians.
    Returns (slide_df, patient_df).
    """
    metric_cols = [c for c in fov_comp_df.columns if c.startswith("frac_")]

    slide_df = (
        fov_comp_df
        .groupby(["disease", "patient", "slide", "radius_um"], observed=True)[metric_cols]
        .median()
        .reset_index()
    )

    patient_df = (
        slide_df
        .groupby(["disease", "patient", "radius_um"], observed=True)[metric_cols]
        .median()
        .reset_index()
    )

    return slide_df, patient_df


In [ ]:
mask = (
    (wtx_uc_inf.obs["slide"] == "UC_batch1_slide2") &
    (wtx_uc_inf.obs["fov"] == '32')
)

wtx_uc_inf.obs.loc[mask, "patient"] = "UC_P2"

In [ ]:
wtx_uc_inf[(wtx_uc_inf.obs['patient']=='UC_P1') &(wtx_uc_inf.obs['slide']=='UC_batch1_slide2')  ].obs['fov']

In [ ]:
# 1) how many (slide,fov) per slide contributed at radius=100?
(fov_comp_df[fov_comp_df["radius_um"]==100]
 .groupby("slide")["fov_uid"].nunique()
 .sort_values(ascending=False)
)

In [ ]:
# 2) are any fov ids shared across slides? (this will still be true, but now it's safe)
(fov_comp_df[["slide","fov"]].drop_duplicates()
 .groupby("fov")["slide"].nunique()
 .sort_values(ascending=False)
 .head(10)
)


In [ ]:

slide_df, patient_df = aggregate_fov_to_slide_patient(fov_comp_df)


In [ ]:
fov_comp_df.to_csv('/path/to/cosmx_data/rec_uc_ile_cd_ogn_fib_imm_neighbor_wtx_comparison.csv')

In [ ]:
fov_comp_df['slide'].value_counts()

In [ ]:


def compute_background_immune_fraction(
    adata,
    shared_imm,
    celltype_col="celltype_aligned",
    fov_col="fov",
    slide_col="slide",
):
    """
    Compute background immune fractions per (slide, fov).
    Returns DataFrame with columns:
      slide, fov, fov_uid, frac_<immune_type>_bg
    """
    rows = []
    immune_set = set(shared_imm)

    gb = adata.obs.groupby([slide_col, fov_col], observed=True)
    groups = gb.indices  # dict: (slide, fov) -> integer positions

    for (slide_id, fov_id), pos in groups.items():
        ad_fov = adata[pos, :]  # safe even if obs_names duplicated

        immune_mask = ad_fov.obs[celltype_col].isin(immune_set)
        n_immune = int(immune_mask.sum())
        if n_immune == 0:
            continue

        counts = ad_fov.obs.loc[immune_mask, celltype_col].value_counts()

        row = {
            "slide": slide_id,
            "fov": fov_id,
            "fov_uid": f"{slide_id}__{fov_id}",
        }
        for ct in shared_imm:
            row[f"frac_{ct}_bg"] = counts.get(ct, 0) / n_immune

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:

def compute_rspo3_enrichment(
    fov_comp_df: pd.DataFrame,
    bg_df: pd.DataFrame,
    immune_type: str,
    eps: float = 1e-4,
    merge_cols=("disease", "fov_uid"),  # safest
):
    """
    Compute log enrichment of immune_type near RSPO3⁺ fibroblasts vs background.

    Enrichment = log((near + eps) / (bg + eps))

    Keeps zeros (important for small radii). Only drops rows where background is missing.
    Returns one row per (disease, patient, slide, fov_uid, radius_um).
    """
    col_near = f"frac_{immune_type}"
    col_bg   = f"frac_{immune_type}_bg"

    # sanity checks
    for c in list(merge_cols) + [col_near]:
        if c not in fov_comp_df.columns:
            raise KeyError(f"fov_comp_df missing column: {c}")
    for c in list(merge_cols) + [col_bg]:
        if c not in bg_df.columns:
            raise KeyError(f"bg_df missing column: {c}")

    df = fov_comp_df.merge(bg_df[list(merge_cols) + [col_bg]], on=list(merge_cols), how="left")

    # require background exists; allow bg==0 and near==0
    df = df[df[col_bg].notna()].copy()

    df["enrichment"] = np.log((df[col_near].fillna(0) + eps) / (df[col_bg] + eps))

    keep = ["disease", "patient", "slide", "FOV", "fov_uid", "radius_um", "enrichment", col_near, col_bg]
    keep = [c for c in keep if c in df.columns]  # in case you didn't keep some cols
    return df[keep]


In [ ]:
def aggregate_enrichment(enrich_df: pd.DataFrame):
    """
    Aggregate FOV-level enrichment -> tissue(slide) -> patient using medians.

    Assumes enrich_df has:
      disease, patient, slide, radius_um, enrichment, and ideally fov_uid.

    Returns
    -------
    slide_df, patient_df
    """
    required = {"disease", "patient", "slide", "radius_um", "enrichment"}
    missing = required - set(enrich_df.columns)
    if missing:
        raise KeyError(f"enrich_df missing columns: {sorted(missing)}")

    df = enrich_df.copy()

    # ✅ If fov_uid exists, dedupe to one row per FOV per radius (safest)
    if "fov_uid" in df.columns:
        df = (
            df.sort_values(["disease", "patient", "slide", "fov_uid", "radius_um"])
              .drop_duplicates(subset=["disease", "fov_uid", "radius_um"], keep="first")
        )

        # slide-level: median across FOVs within each slide
        slide_df = (
            df.groupby(["disease", "patient", "slide", "radius_um"], observed=True)["enrichment"]
              .median()
              .reset_index()
        )
    else:
        # fallback: assume each row already unique per FOV
        slide_df = (
            df.groupby(["disease", "patient", "slide", "radius_um"], observed=True)["enrichment"]
              .median()
              .reset_index()
        )

    # patient-level: median across slides within each patient
    patient_df = (
        slide_df.groupby(["disease", "patient", "radius_um"], observed=True)["enrichment"]
                .median()
                .reset_index()
    )

    return slide_df, patient_df


In [ ]:
# UC background
bg_uc = compute_background_immune_fraction(
    wtx_uc_inf,
    shared_imm=shared_imm,
    celltype_col="celltype_aligned",
    fov_col="fov",
    slide_col="slide",
)
bg_uc["disease"] = "UC"

# CD background
bg_cd = compute_background_immune_fraction(
    wtx_cd_inf,
    shared_imm=shared_imm,
    celltype_col="celltype_aligned",
    fov_col="fov",
    slide_col="slide",
)
bg_cd["disease"] = "CD"

# combine
bg_df = pd.concat([bg_uc, bg_cd], ignore_index=True)


In [ ]:
enrich_treg = compute_rspo3_enrichment(
    fov_comp_df=fov_comp_df,
    bg_df=bg_df,
    immune_type="Treg",
    eps=1e-4,   # recommended for fractions
)


In [ ]:
# Aggregate
slide_enrich_treg, patient_enrich_treg = aggregate_enrichment(enrich_treg)

patient_enrich_treg[patient_enrich_treg["radius_um"] == 100]


In [ ]:
# Enrichment for Treg
enrich_mac_s_m_s = compute_rspo3_enrichment(
    fov_comp_df,
    bg_df,
    immune_type="Mac S+M+S+"
)


In [ ]:
# Enrichment for Treg
enrich_inf_mo_mac = compute_rspo3_enrichment(
    fov_comp_df,
    bg_df,
    immune_type="Inflammatory Mono/Mac"
)


In [ ]:

def bootstrap_patient_value(
    df,                      # must contain patient, slide, radius_um, value_col
    patient_id,
    radius_um,
    value_col,
    n_boot=1000,
):
    sub = df[(df["patient"] == patient_id) & (df["radius_um"] == radius_um)]

    slides = sub["slide"].unique()
    boot = []

    for _ in range(n_boot):
        slide_vals = []
        for s in slides:
            vals = sub[sub["slide"] == s][value_col].values
            if len(vals) == 0:
                continue
            resampled = np.random.choice(vals, size=len(vals), replace=True)
            slide_vals.append(np.median(resampled))
        if len(slide_vals) > 0:
            boot.append(np.median(slide_vals))

    return np.array(boot)


In [ ]:
radius = 100
immune = "Treg"

# enrich_treg is your FOV-level enrichment df with column "enrichment"
for p in enrich_treg["patient"].unique():
    boot = bootstrap_patient_value(
        enrich_treg, patient_id=p, radius_um=radius, value_col="enrichment"
    )
    print(p, np.percentile(boot, [5, 50, 95]))

In [ ]:
radius = 100
immune = "Treg"

# bootstrap on fov_comp_df for raw fraction
fov_raw = fov_comp_df.copy()
value_col = f"frac_{immune}"

for p in fov_raw["patient"].unique():
    boot = bootstrap_patient_value(
        fov_raw, patient_id=p, radius_um=radius, value_col=value_col
    )
    print(p, np.percentile(boot, [5, 50, 95]))


In [ ]:
radius = 100
immune = "Inflammatory Mono/Mac"

# enrich_treg is your FOV-level enrichment df with column "enrichment"
for p in enrich_inf_mo_mac["patient"].unique():
    boot = bootstrap_patient_value(
        enrich_treg, patient_id=p, radius_um=radius, value_col="enrichment"
    )
    print(p, np.percentile(boot, [5, 50, 95]))

In [ ]:
radius = 100
immune = "Inflammatory Mono/Mac"

# bootstrap on fov_comp_df for raw fraction
fov_raw = fov_comp_df.copy()
value_col = f"frac_{immune}"

for p in fov_raw["patient"].unique():
    boot = bootstrap_patient_value(
        fov_raw, patient_id=p, radius_um=radius, value_col=value_col
    )
    print(p, np.percentile(boot, [5, 50, 95]))


In [ ]:
radius = 100
immune = "Mac S+M+S+"

# enrich_treg is your FOV-level enrichment df with column "enrichment"
for p in enrich_mac_s_m_s["patient"].unique():
    boot = bootstrap_patient_value(
        enrich_treg, patient_id=p, radius_um=radius, value_col="enrichment"
    )
    print(p, np.percentile(boot, [5, 50, 95]))

In [ ]:


cell_types = ["Treg", "Inflammatory Mono/Mac", "Mac S+M+S+"]
radius_use = 100  # you said you’re focusing on 100 um


In [ ]:
def make_long_raw(fov_comp_df, cell_types, radius_um=100):
    cols = [f"frac_{ct}" for ct in cell_types]
    df = fov_comp_df[fov_comp_df["radius_um"] == radius_um].copy()

    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in fov_comp_df: {missing}")

    long = df.melt(
        id_vars=["disease", "patient", "slide", "FOV", "radius_um"],
        value_vars=cols,
        var_name="cell_type",
        value_name="value"
    )
    long["cell_type"] = long["cell_type"].str.replace("^frac_", "", regex=True)
    return long

raw_long = make_long_raw(fov_comp_df, cell_types, radius_um=radius_use)
raw_long.head()


In [ ]:
fov_comp_df['patient_slide']=[i+'_'+j.split('slide')[1] for i,j in zip(fov_comp_df['patient'].tolist(),fov_comp_df['slide'].tolist())]

In [ ]:
fov_comp_df[['patient','slide']].value_counts()

In [ ]:

def plot_fov_by_slide(
    fov_comp_df,
    cell_type,
    radius_um=100,
    y_label="Raw fraction among immune neighbors",
):
    col = f"frac_{cell_type}"
    df = fov_comp_df[fov_comp_df["radius_um"] == radius_um].copy()

    diseases = ["UC", "CD"]
    x_pos = {d: i for i, d in enumerate(diseases)}

    # Split slides by disease
    uc_slides = df.loc[df["disease"] == "UC", "patient_slide"].unique()
    cd_slides = df.loc[df["disease"] == "CD", "patient_slide"].unique()

    # Color palettes
    uc_cmap = plt.cm.Blues
    cd_cmap = plt.cm.Oranges

    uc_colors = uc_cmap(np.linspace(0.4, 0.85, len(uc_slides)))
    cd_colors = cd_cmap(np.linspace(0.4, 0.85, len(cd_slides)))

    slide_to_color = {}
    for s, c in zip(uc_slides, uc_colors):
        slide_to_color[s] = c
    for s, c in zip(cd_slides, cd_colors):
        slide_to_color[s] = c

    fig, ax = plt.subplots(figsize=(4.8, 3.5))

    for _, r in df.iterrows():
        x = x_pos[r["disease"]]
        jitter = (np.random.rand() - 0.5) * 0.5

        ax.scatter(
            x + jitter,
            r[col],
            s=12,
            alpha=0.7,
            color=slide_to_color[r["patient_slide"]],
        )

    ax.set_xticks([0, 1])
    ax.set_xticklabels(diseases)
    ax.set_ylabel(y_label)
    ax.set_title(cell_type)

    # Legend grouped by disease
    handles_uc = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=slide_to_color[s], label=s)
        for s in uc_slides
    ]
    handles_cd = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=slide_to_color[s], label=s)
        for s in cd_slides
    ]

    legend1 = ax.legend(
        handles=handles_uc,
        title="UC slides",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
    )
    ax.add_artist(legend1)

    ax.legend(
        handles=handles_cd,
        title="CD slides",
        bbox_to_anchor=(1.02, 0.45),
        loc="upper left",
        frameon=False,
    )

    plt.tight_layout()
    plt.show()


In [ ]:
#sample_dir = '/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/flat'
sample_dir = '/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/flat' 	

In [ ]:
os.listdir(sample_dir)

In [ ]:
counts = pd.read_csv(sample_dir+'/CD_batch2_dir_exprMat_file.csv', index_col=0)

In [ ]:
metadata = pd.read_csv(sample_dir+'/CD_batch2_dir_metadata_file.csv')      # Indexed by cell_id

In [ ]:
counts_clean = counts.drop(['cell_ID'],axis = 1)

In [ ]:
counts_clean.index = counts_clean['cell']
counts_clean = counts_clean.drop(['cell'],axis = 1)
counts_clean

In [ ]:
# Create AnnData object
adata = sc.AnnData(X=counts_clean)

In [ ]:
# Add metadata to AnnData
adata.obs = metadata
adata

In [ ]:
adata.var['gene'] = adata.var_names.tolist()

In [ ]:
adata.obsm['spatial_fov']=adata.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_probe']=[True if 'SystemControl' in i else False for i in adata.var.index.tolist()]

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]].copy()
adata=adata[:, ~adata.var["control_probe"]].copy()

In [ ]:

if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)


In [ ]:

sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts",'Area'],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=50)
sc.pp.filter_cells(adata, min_genes=25)
sc.pp.filter_cells(adata, max_counts=1000)
sc.pp.filter_cells(adata, max_genes=450)
adata=adata[adata.obs['Area']<=30000]

In [ ]:
adata_inf = adata[adata.obs['fov'].isin(range(1,12))]

In [ ]:
adata_inf.obs[['orig.ident']]='CD_B6_CD_B3'
adata_inf.obs['patient']='CD_B3R'
adata_inf.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata_inf.obs['fov'].tolist(), adata_inf.obs['cell_ID'].tolist())]


In [ ]:
slide=[]
for i,j in zip(adata_inf.obs['patient'].tolist(),adata_inf.obs['fov'].tolist()):
    j= int(j)
    if j <=4:
        slide.append(i+'_'+'slide1')
    else:
        slide.append(i+'_'+'slide3')


In [ ]:
adata_inf.obs['slide']=slide

In [ ]:
adata_inf.layers['raw']=pd.DataFrame(adata_inf.X.toarray())

In [ ]:
sc.pp.normalize_total(adata_inf, inplace=True)
sc.pp.log1p(adata_inf)


In [ ]:
out_path = "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj"
out_file = os.path.join(out_path, "CD_B3r_qc_norm_log1p.h5ad")

# make directory if it doesn't exist
os.makedirs(out_path, exist_ok=True)

# write AnnData
adata_inf.write_h5ad(out_file)


In [ ]:
adata_norm_log1p = pd.DataFrame(adata_inf.X.toarray())

In [ ]:
adata_norm_log1p.columns = adata_inf.var.index.tolist()
adata_norm_log1p.index=adata_inf.obs['fov_cell_id'].tolist()
adata_norm_log1p

In [ ]:
out_path = "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj"
out_file = os.path.join(out_path, "CD_B3r_log1p_norm_counts.csv")

# make directory if it doesn't exist
os.makedirs(out_path, exist_ok=True)

# write AnnData
adata_norm_log1p.to_csv(out_file)


In [ ]:
out_path = "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj"
out_file = os.path.join(out_path, "CD_B1r_log1p_norm_counts.csv")

# make directory if it doesn't exist
os.makedirs(out_path, exist_ok=True)

# write AnnData
adata_norm_log1p.to_csv(out_file)


In [ ]:
adata_norm_nolog1p = adata_inf.copy()
adata_norm_nolog1p.X = adata_inf.layers['raw']

In [ ]:
sc.pp.normalize_total(adata_norm_nolog1p, inplace=True)

In [ ]:
adata_norm_nolog1p_count = pd.DataFrame(adata_norm_nolog1p.X)

In [ ]:
adata_norm_nolog1p_count.to_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj/CD_B1_norm_counts.csv')

In [ ]:
adata_norm_nolog1p_count.to_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj/CD_B3_norm_counts.csv')

In [ ]:
CD_B1r_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj/CD_B1r_aucell_anno.csv')

In [ ]:
CD_B3r_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj/CD_B3r_aucell_anno.csv')

In [ ]:
adata_CD_B3r=sc.read_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj/CD_B3r_qc_norm_log1p.h5ad")
adata_CD_B1r=sc.read_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj/CD_B1r_qc_norm_log1p.h5ad")

In [ ]:
mapper= pd.read_csv('/path/to/scrna/cd/cell_type_category_map.csv')

In [ ]:
# Step 1: Clean the names
mapper['cell_type_aucell'] = mapper['cell type'].str.replace(r'[+\-\/()]', ' ', regex=True)
mapper['cell_type_aucell'] = mapper['cell_type_aucell'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Step 2: Disambiguate duplicates by appending " 1", " 2", etc.
mapper['cell_type_aucell'] = (
    mapper.groupby('cell_type_aucell').cumcount().astype(str).replace('0', '', regex=False)
    .radd(mapper['cell_type_aucell'] + ' ').str.strip()
)

In [ ]:
CD_B1r_anno['label_clean'] = CD_B1r_anno['label'].str.rstrip()
CD_B3r_anno['label_clean'] = CD_B3r_anno['label'].str.rstrip()

In [ ]:
CD_B1r_anno_map=CD_B1r_anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')
CD_B3r_anno_map=CD_B3r_anno.merge(mapper, how = 'left', left_on = 'label_clean',right_on = 'cell_type_aucell')

In [ ]:
adata_CD_B3r.obs['cell_type'] = CD_B3r_anno_map['cell type'].tolist()
adata_CD_B3r.obs['cell_type_short'] = CD_B3r_anno_map['cell type short'].tolist()
adata_CD_B3r.obs['cell_category'] = CD_B3r_anno_map['cell category'].tolist()
adata_CD_B3r.obs['cell_type_general'] = CD_B3r_anno_map['cell type general'].tolist()

In [ ]:
adata_CD_B1r.obs['cell_type'] = CD_B1r_anno_map['cell type'].tolist()
adata_CD_B1r.obs['cell_type_short'] = CD_B1r_anno_map['cell type short'].tolist()
adata_CD_B1r.obs['cell_category'] = CD_B1r_anno_map['cell category'].tolist()
adata_CD_B1r.obs['cell_type_general'] = CD_B1r_anno_map['cell type general'].tolist()

In [ ]:
original_cats = list(adata_CD_B1r.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in original_cats if cat != 'Cycling'] + ['Cycling']

adata_CD_B1r.obs['cell_category'] = pd.Categorical(
    adata_CD_B1r.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
)

In [ ]:
original_cats = list(adata_CD_B3r.obs['cell_category'].astype('category').cat.categories)
reordered_cats = [cat for cat in original_cats if cat != 'Cycling'] + ['Cycling']

adata_CD_B3r.obs['cell_category'] = pd.Categorical(
    adata_CD_B3r.obs['cell_category'],
    categories=reordered_cats,
    ordered=True
)

In [ ]:
adata_CD_B3r.write_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj/CD_B3r_anno.h5ad")
adata_CD_B1r.write_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj/CD_B1r_anno.h5ad")

In [ ]:
adata_CD_B3r=sc.read_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch2_dir/h5ad_obj/CD_B3r_anno.h5ad")
adata_CD_B1r=sc.read_h5ad( "/path/to/cosmx_data/uc_serial_cuts/1000plex/CD_batch1_dir/h5ad_obj/CD_B1r_anno.h5ad")

In [ ]:
adata_col_cd = sc.concat(
    [adata_CD_B1r, adata_CD_B3r],
    join="inner"
)

In [ ]:
# cd
adata_col_cd.obs["celltype_aligned"] = align_labels(
    adata_col_cd.obs["cell_type_short"].tolist(),
    collapse_prolif=True
)


In [ ]:

RSPO3_FIB_LABEL = "OGN+RSPO3+ Fib"
CONNECTIVE_LABEL = "Connective"

def compute_rspo3_fib_neighbor_composition(
    adatas: dict,
    shared_imm: list,
    r_list: list,
    celltype_col: str = "celltype_aligned",
    fov_col: str = "fov",
    slide_col: str = "slide",
    patient_col: str = "patient",
    spatial_key: str = "spatial_fov",
    # category columns: will auto-pick whichever exists
    category_cols_priority: tuple = ("cell_category", "aucell_cell_category_from_type"),
    # optional: if you have a column encoding region (colon/ileum) use it
    region_col: str | None = None,
) -> pd.DataFrame:
    """
    For each AnnData in `adatas` and each (slide, fov):
      - centers: Connective AND OGN+RSPO3+ Fib
      - neighbors: cells within radius R
      - counts immune neighbors by `celltype_col` in `shared_imm`
      - outputs per-(slide,fov,radius) immune composition fractions

    Adds:
      - condition: key in `adatas` dict
      - disease: parsed from condition if startswith 'UC'/'CD' else condition
      - region: parsed from condition if contains 'colon'/'ileum' else from region_col if provided
    """

    immune_label_set = list(shared_imm)
    immune_set = set(immune_label_set)

    results = []

    for condition, adata in adatas.items():
        # basic checks
        assert spatial_key in adata.obsm, f"{condition}: missing obsm['{spatial_key}']"
        for col in [fov_col, slide_col, patient_col, celltype_col]:
            assert col in adata.obs.columns, f"{condition}: missing obs['{col}']"

        # pick a category column that exists
        cat_col = None
        for c in category_cols_priority:
            if c in adata.obs.columns:
                cat_col = c
                break
        if cat_col is None:
            raise KeyError(f"{condition}: none of category cols found: {category_cols_priority}")

        # disease/region labels
        cond_lower = str(condition).lower()
        disease = "UC" if cond_lower.startswith("uc") else ("CD" if cond_lower.startswith("cd") else str(condition))

        if region_col is not None and region_col in adata.obs.columns:
            # take region from obs (should be constant within adata)
            region = str(adata.obs[region_col].iloc[0])
        else:
            # parse from condition key if possible
            if "ileum" in cond_lower or "ileal" in cond_lower:
                region = "ileum"
            elif "colon" in cond_lower or "colonic" in cond_lower:
                region = "colon"
            else:
                region = None

        # group by (slide, fov) using integer positions (safe if obs_names not unique)
        gb = adata.obs.groupby([slide_col, fov_col], observed=True)
        groups = gb.indices  # (slide, fov) -> np.ndarray[int]

        for (slide_id, fov_id), pos in groups.items():
            ad_fov = adata[pos, :].copy()

            ct_series = ad_fov.obs[celltype_col].astype(str).str.strip()
            cat_series = ad_fov.obs[cat_col].astype(str).str.strip()

            centers_mask = cat_series.eq(CONNECTIVE_LABEL) & ct_series.eq(RSPO3_FIB_LABEL)
            n_centers = int(centers_mask.sum())
            if n_centers == 0:
                continue

            XY = np.asarray(ad_fov.obsm[spatial_key])
            if XY.ndim != 2 or XY.shape[1] != 2:
                raise ValueError(f"{condition} slide={slide_id} fov={fov_id}: spatial must be (n,2)")

            immune_mask = ct_series.isin(immune_set).to_numpy()
            tree = cKDTree(XY)
            center_idx = np.where(centers_mask.to_numpy())[0]

            patient_val = ad_fov.obs[patient_col].iloc[0]
            fov_uid = f"{slide_id}__{fov_id}"

            for R in r_list:
                total_counts = {ct: 0 for ct in immune_label_set}
                total_immune_neighbors = 0

                for i in center_idx:
                    nbrs = tree.query_ball_point(XY[i], r=R)
                    # drop self
                    if nbrs:
                        nbrs = [j for j in nbrs if j != i]

                    for j in nbrs:
                        if immune_mask[j]:
                            ct = ct_series.iloc[j]
                            total_counts[ct] += 1
                            total_immune_neighbors += 1

                if total_immune_neighbors > 0:
                    comp = {f"frac_{ct}": total_counts[ct] / total_immune_neighbors for ct in immune_label_set}
                else:
                    comp = {f"frac_{ct}": 0.0 for ct in immune_label_set}

                results.append({
                    "condition": condition,   # e.g. UC_colon, CD_ileum, CD_colon
                    "disease": disease,       # UC / CD
                    "region": region,         # colon / ileum / None
                    "patient": patient_val,
                    "slide": slide_id,
                    "fov": fov_id,
                    "fov_uid": fov_uid,
                    "radius_um": R,
                    "n_cells": ad_fov.n_obs,
                    "n_rspo3_fib_centers": n_centers,
                    "n_immune_neighbors_total": total_immune_neighbors,
                    **comp,
                })

    return pd.DataFrame(results)


In [ ]:
R_LIST = [50, 100, 200]

adatas = {
    "UC_colon": wtx_uc_inf,
    "CD_ileum": wtx_cd_inf,
    "CD_colon": adata_col_cd,   # your 1000-plex (imputed or annotated)
}

fov_comp_df = compute_rspo3_fib_neighbor_composition(
    adatas=adatas,
    shared_imm=shared_imm,
    r_list=R_LIST,
    celltype_col="celltype_aligned",
    fov_col="fov",
    slide_col="slide",
    patient_col="patient",
    spatial_key="spatial_fov",
)


In [ ]:
# 1) how many (slide,fov) per slide contributed at radius=100?
(fov_comp_df[fov_comp_df["radius_um"]==100]
 .groupby("slide")["fov_uid"].nunique()
 .sort_values(ascending=False)
)



In [ ]:
fov_comp_df.to_csv('/path/to/cosmx_data/rec_uc_ile_col_cd_ogn_fib_imm_neighbor_wtx_comparison.csv')

In [ ]:

slide_df, patient_df = aggregate_fov_to_slide_patient(fov_comp_df)


In [ ]:


cell_types = ["Treg", "Inflammatory Mono/Mac", "Mac S+M+S+"]
radius_use = 100  # you said you’re focusing on 100 um


In [ ]:
def make_long_raw(fov_comp_df, cell_types, radius_um=100, region_col="region"):
    cols = [f"frac_{ct}" for ct in cell_types]
    df = fov_comp_df[fov_comp_df["radius_um"] == radius_um].copy()

    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in fov_comp_df: {missing}")

    if region_col not in df.columns:
        raise KeyError(f"Missing '{region_col}' column in fov_comp_df")

    long = df.melt(
        id_vars=["disease", region_col, "patient", "slide", "fov", "radius_um"],
        value_vars=cols,
        var_name="cell_type",
        value_name="value"
    )
    long["cell_type"] = long["cell_type"].str.replace("^frac_", "", regex=True)
    return long

raw_long = make_long_raw(fov_comp_df, cell_types, radius_um=radius_use, region_col="region")
raw_long.head()


In [ ]:
def agg_to_slide(long_df):
    slide_df = (long_df
        .groupby(["disease", 'region',"patient", "slide", "radius_um", "cell_type"], observed=True)["value"]
        .median()
        .reset_index()
    )
    return slide_df

raw_slide = agg_to_slide(raw_long)
raw_slide.head()


In [ ]:
fov_comp_df['patient_slide']=[i+'_'+j.split('slide')[1] for i,j in zip(fov_comp_df['patient'].tolist(),fov_comp_df['slide'].tolist())]

In [ ]:
fov_comp_df[['patient','slide']].value_counts()

In [ ]:
def compute_weighted_slide_means(
    fov_comp_df,
    cell_types,
    radius_um=200,
    weight_col="n_immune_neighbors_total",
):
    rows = []

    df = fov_comp_df[fov_comp_df["radius_um"] == radius_um].copy()

    for ct in cell_types:
        frac_col = f"frac_{ct}"

        if frac_col not in df.columns:
            raise KeyError(f"Missing column {frac_col}")

        slide_df = (
            df
            .groupby(["disease", "region", "slide"], observed=True)
            .apply(
                lambda g: np.average(
                    g[frac_col],
                    weights=g[weight_col]
                ) if g[weight_col].sum() > 0 else np.nan
            )
            .reset_index(name="weighted_frac")
        )

        slide_df["cell_type"] = ct
        rows.append(slide_df)

    return pd.concat(rows, ignore_index=True)


In [ ]:
cell_types = [
    "Treg",
    "Inflammatory Mono/Mac",
    "Mac S+M+S+"
]

slide_weighted = compute_weighted_slide_means(
    fov_comp_df,
    cell_types=cell_types,
    radius_um=200
)

slide_weighted


In [ ]:

def plot_fov_with_slide_weighted_and_slide_legend(
    fov_comp_df,
    slide_weighted_df,
    cell_type,
    radius_um=200,
    region_col="region",
    slide_col="slide",
    y_label="Fraction among immune neighbors",
    show_iqr=True,
):
    frac_col = f"frac_{cell_type}"
    sw_col = "weighted_frac"  # column in slide_weighted_df

    df = fov_comp_df[fov_comp_df["radius_um"] == radius_um].copy()
    sw = slide_weighted_df[slide_weighted_df["cell_type"] == cell_type].copy()

    # checks
    for c in ["disease", region_col, slide_col]:
        if c not in df.columns:
            raise KeyError(f"Missing '{c}' in fov_comp_df")
        if c not in sw.columns:
            raise KeyError(f"Missing '{c}' in slide_weighted_df")

    if frac_col not in df.columns:
        raise KeyError(f"Missing '{frac_col}' in fov_comp_df")
    if sw_col not in sw.columns:
        raise KeyError(f"Missing '{sw_col}' in slide_weighted_df")

    # group labels
    df["group"] = df["disease"] + "_" + df[region_col].astype(str)
    sw["group"] = sw["disease"] + "_" + sw[region_col].astype(str)

    group_order = ["UC_colon", "CD_colon", "CD_ileum"]
    group_order = [g for g in group_order if g in df["group"].unique()]
    x_pos = {g: i for i, g in enumerate(group_order)}

    # color mapping: (group, slide) -> color
    palette_map = {
        "UC_colon": plt.cm.Blues,
        "CD_colon": plt.cm.Oranges,
        "CD_ileum": plt.cm.Greens,
    }

    slide_to_color = {}
    for g in group_order:
        slides_g = df.loc[df["group"] == g, slide_col].unique()
        cmap = palette_map.get(g, plt.cm.Greys)
        colors = cmap(np.linspace(0.45, 0.85, max(len(slides_g), 1)))
        for s, c in zip(slides_g, colors):
            slide_to_color[(g, s)] = c

    fig, ax = plt.subplots(figsize=(6.6, 3.8))

    # 1) FOV dots (faint)
    for _, r in df.iterrows():
        g = r["group"]
        x = x_pos[g]
        jitter = (np.random.rand() - 0.5) * 0.5
        ax.scatter(
            x + jitter,
            r[frac_col],
            s=10,
            alpha=0.35,
            color=slide_to_color[(g, r[slide_col])],
            linewidths=0,
        )

    # 2) Slide weighted mean (bold) + optional IQR whisker from FOVs
    for _, r in sw.iterrows():
        g = r["group"]
        x = x_pos[g]
        s = r[slide_col]
        color = slide_to_color.get((g, s), "k")

        if show_iqr:
            sub = df[(df["group"] == g) & (df[slide_col] == s)][frac_col].dropna()
            if len(sub) > 0:
                q1, q3 = np.percentile(sub, [25, 75])
                ax.plot([x, x], [q1, q3], color=color, linewidth=2, alpha=0.9, zorder=4)

        ax.scatter(
            x,
            r[sw_col],
            s=30,
            color=color,
            edgecolor="black",
            linewidth=0.6,
            zorder=5,
        )

    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(group_order, rotation=25, ha="right")
    ax.set_ylabel(y_label)
    ax.set_title(f"OGN+RSPO3+ Fibroblast Neighbors\n{cell_type} (radius={radius_um}µm) ")

    # 3) Slide legend (matches colors)
    legend_handles = []
    seen = set()
    for g in group_order:
        slides_g = df.loc[df["group"] == g, slide_col].unique()
        for s in slides_g:
            key = (g, s)
            if key in seen:
                continue
            seen.add(key)
            legend_handles.append(
                plt.Line2D([0], [0], marker="o", linestyle="",
                           color=slide_to_color[(g, s)],
                           label=f"{g}: {s}", markersize=6)
            )

    ax.legend(
        handles=legend_handles,
        title="Slide",
        bbox_to_anchor=(1.02, 1.05),
        loc="upper left",
        frameon=False,
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_fov_with_slide_weighted_and_slide_legend(
    fov_comp_df=fov_comp_df,
    slide_weighted_df=slide_weighted,
    cell_type="Treg",
    radius_um=200,
    region_col="region",
    slide_col="slide",
)


In [ ]:
plot_fov_with_slide_weighted_and_slide_legend(
    fov_comp_df=fov_comp_df,
    slide_weighted_df=slide_weighted,
    cell_type="Mac S+M+S+",
    radius_um=200,
    region_col="region",
    slide_col="slide",
)


In [ ]:

# ── 1. Define groupings ──────────────────────────────────────────────────────
group_map = {
    "CD8 GZMK+ Trm": "CD8 Trm",
    "CD8 GZMK- Trm": "CD8 Trm",
    "Gd T":           "Gd T",
    "Gd T (Vd2g9+)":  "Gd T",
    "Mac S+SG+":      "Mac S+",
    "Mac S+XS-":      "Mac S+",
    "cDC1":           "cDC",
    "cDC2":           "cDC",
}

individual = [
    "Mac S+M+P+", "Mac S+M+S+", "Mono/Mac", "Inflammatory Mono/Mac",
    "Trans cDC2/Mono/Mac", "Treg", "Tfh", "B cell", "CCR7+ DC",
    "CD4 Effector", "CD4 Trm", "Granulocyte", "Neutrophil", "Resting T"
]

# ── 2. Prepare fov_comp_df ───────────────────────────────────────────────────
radius = 100
df = fov_comp_df[fov_comp_df["radius_um"] == radius].copy()
df = df[df["n_rspo3_fib_centers"] > 0].copy()

# add grouped columns
for members, group_name in [
    (["CD8 GZMK+ Trm", "CD8 GZMK- Trm"], "CD8 Trm"),
    (["Gd T", "Gd T (Vd2g9+)"],           "Gd T"),
    (["Mac S+SG+", "Mac S+XS-"],           "Mac S+"),
    (["cDC1", "cDC2"],                     "cDC"),
]:
    cols = [f"frac_{m}" for m in members if f"frac_{m}" in df.columns]
    df[f"frac_{group_name}"] = df[cols].sum(axis=1)

# final cell type list to plot
plot_celltypes = individual + ["CD8 Trm", "Gd T", "Mac S+", "cDC"]

# ── 3. Patient-level aggregation ─────────────────────────────────────────────
frac_cols = [f"frac_{ct}" for ct in plot_celltypes if f"frac_{ct}" in df.columns]
patient_df = (
    df.groupby(["condition", "patient"])[frac_cols]
    .mean()
    .reset_index()
)

# melt to long form
plot_df = df.melt(
    id_vars=["condition", "patient"],
    value_vars=frac_cols,
    var_name="celltype",
    value_name="fraction",
)
plot_df["celltype"] = plot_df["celltype"].str.replace("frac_", "", regex=False)

# ── 4. Plot ───────────────────────────────────────────────────────────────────
condition_order = ["UC_colon", "CD_colon", "CD_ileum"]
palette = {"UC_colon": "#4393c3", "CD_colon": "#f4a582", "CD_ileum": "#d6604d"}

n_celltypes = len(plot_celltypes)
ncols = 4
nrows = int(np.ceil(n_celltypes / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3), sharey=False)
axes = axes.flatten()

for ax, ct in zip(axes, plot_celltypes):
    sub = plot_df[plot_df["celltype"] == ct]
    sns.boxplot(
        data=sub, x="condition", y="fraction", order=condition_order,
        palette=palette, width=0.5, showcaps=False,
        boxprops=dict(alpha=0.5), whiskerprops=dict(alpha=0.5),
        flierprops=dict(marker=""), ax=ax
    )
    sns.stripplot(
        data=sub, x="condition", y="fraction", order=condition_order,
        palette=palette, size=5, jitter=True, ax=ax, zorder=3
    )
    ax.set_title(ct, fontsize=9)
    ax.set_xlabel("")
    ax.set_xticklabels(["UC", "CD\ncolon", "CD\nileum"], fontsize=7)
    ax.set_ylabel("Neighbor fraction", fontsize=7)

# hide unused axes
for ax in axes[n_celltypes:]:
    ax.set_visible(False)

plt.suptitle(f"OGN+RSPO3+ Fib neighbor composition (r={radius}µm)", fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig("rspo3_fib_neighbor_boxplots.pdf", bbox_inches="tight")
plt.show()

In [ ]:


# ── 1. Subset to OGN+RSPO3+ Fib from each condition ─────────────────────────
fib_adatas = {}
for cond, adata in adatas.items():
    fib = adata[adata.obs["celltype_aligned"] == "OGN+RSPO3+ Fib"].copy()
    fib.obs["condition"] = cond
    fib_adatas[cond] = fib

# ── 2. Merge ──────────────────────────────────────────────────────────────────
fib_combined = sc.concat(
    fib_adatas,
    label="condition",
    merge="same",
)

print(fib_combined)
print(fib_combined.obs["condition"].value_counts())

In [ ]:
# ── 3. Preprocess ─────────────────────────────────────────────────────────────
sc.pp.normalize_total(fib_combined, target_sum=1e4)
sc.pp.log1p(fib_combined)
sc.pp.highly_variable_genes(fib_combined, n_top_genes=300, batch_key="condition")
fib_combined = fib_combined[:, fib_combined.var["highly_variable"]].copy()
sc.pp.pca(fib_combined, n_comps=30)

# ── 4. Harmony integration ────────────────────────────────────────────────────
sc.external.pp.harmony_integrate(fib_combined, key="condition", basis="X_pca", adjusted_basis="X_pca_harmony",  )

# ── 5. UMAP ───────────────────────────────────────────────────────────────────
sc.pp.neighbors(fib_combined, use_rep="X_pca_harmony", n_neighbors=5)
sc.tl.umap(fib_combined)

# ── 6. Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.umap(fib_combined, color="condition", 
           palette={"UC_colon": "#4393c3", "CD_colon": "#f4a582", "CD_ileum": "#d6604d"},
           title="OGN+RSPO3+ Fib by condition", ax=axes[0], show=False)

# if you have a patient col in obs
sc.pl.umap(fib_combined, color="patient", title="OGN+RSPO3+ Fib by patient", 
           ax=axes[1], show=False)

plt.tight_layout()
plt.savefig("rspo3_fib_harmony_umap.pdf", bbox_inches="tight")
plt.show()

In [ ]:

# use harmony embedding
X = fib_combined.obsm["X_pca_harmony"]
labels = fib_combined.obs["condition"].values

# train on UC and ileal CD only, predict colonic CD
train_mask = np.isin(labels, ["UC_colon", "CD_ileum"])
test_mask  = labels == "CD_colon"

knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X[train_mask], labels[train_mask])
pred = knn.predict(X[test_mask])

print(pd.Series(pred).value_counts(normalize=True))

In [ ]:
# ── condition counts for legend labels ───────────────────────────────────────
cond_counts = fib_combined.obs["condition"].value_counts()
pred_counts  = pd.Series(pred).value_counts()

# remap condition names in obs for display
fib_combined.obs["condition_label"] = fib_combined.obs["condition"].map({
    "UC_colon":  f"UC_colon ({cond_counts['UC_colon']})",
    "CD_colon":  f"CD_colon ({cond_counts['CD_colon']})",
    "CD_ileum":  f"CD_ileum ({cond_counts['CD_ileum']})",
}).astype("category")

# remap predicted labels
fib_combined.obs["knn_pred_label"] = "background"
fib_combined.obs.loc[fib_combined.obs["condition"] == "CD_colon", "knn_pred_label"] = (
    pd.Series(pred).map({
        "UC_colon":  f"UC_colon ({pred_counts.get('UC_colon', 0)})",
        "CD_ileum":  f"CD_ileum ({pred_counts.get('CD_ileum', 0)})",
    }).values
)
fib_combined.obs["knn_pred_label"] = fib_combined.obs["knn_pred_label"].astype("category")

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10,4))

sc.pl.umap(
    fib_combined,
    size=35,
    color="condition_label",
    palette={
        f"UC_colon ({cond_counts['UC_colon']})":  "#4393c3",
        f"CD_colon ({cond_counts['CD_colon']})":  "#f4a582",
        f"CD_ileum ({cond_counts['CD_ileum']})":  "#d6604d",
    },
    title="OGN+RSPO3+ Fib by condition",
    ax=axes[0],
    show=False,
)

sc.pl.umap(
    fib_combined,
    size=35,
    color="knn_pred_label",
    palette={
        f"UC_colon ({pred_counts.get('UC_colon', 0)})":  "#4393c3",
        f"CD_ileum ({pred_counts.get('CD_ileum', 0)})":  "#d6604d",
        "background": "#d3d3d3",
    },
    title="Colonic CD Fib predicted identity",
    ax=axes[1],
    show=False,
)

plt.tight_layout()
plt.savefig("rspo3_fib_knn_umap.pdf", bbox_inches="tight")
plt.show()